# AELIONIX BLACKFORGE — Phase 6 Colab Validation

This notebook performs a deterministic, one-click validation of the Blackforge **Web & API Security Capability Foundation** (Phase 6).

**What this validates:**
- Repository integrity and commit verification
- Dependency installation (runtime + dev extras)
- All Blackforge imports, including the new `blackforge.webapi` modules
- Full automated test suite (webapi phase included)
- Bootstrap + health verification, including the new `webapi_ready` check
- Ten typed web/api security capabilities (application discovery, endpoint enumeration, API surface discovery, security-header analysis, cookie analysis, CORS analysis, authentication-surface observation, OpenAPI review, GraphQL discovery, request/response observation)
- The full webapi pipeline: capability -> mock transport -> normalization -> evidence (artifact + typed observations, `DERIVED_FROM`) -> World Model -> best-effort memory link
- IDEMPOTENT reruns and mission isolation across fresh SQLite connections
- **No generic execution surface**: only the ten typed capability contracts exist; unknown capabilities are rejected
- Authorization enforced *before* any transport execution (out-of-scope targets and out-of-scope capabilities are denied)
- GET-only request/response observation with redaction at the boundary
- Failure-aware statuses (RATE_LIMITED, REQUEST_FAILED, NO_EVIDENCE, LIMITED, PARTIAL, SUCCESS)
- Confidence policy: PASSIVE→LOW; direct ACTIVE kinds→HIGH; document kinds→MEDIUM
- World Model semantics: APPLICATION named by hostname; ENDPOINT/API named by URL; analysis assertions bound to the host APPLICATION; request/response assertions bound to ENDPOINT


---


In [ ]:
import sys
import platform

print("Blackforge Phase 6 Colab Validation (Web & API Security Capability Foundation)")
print("=" * 60)
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("=" * 60)

assert sys.version_info >= (3, 10), f"Blackforge requires Python 3.10+, got {sys.version}"
print("Python version check: PASS")


---


In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import shutil

# ── Configuration (edit here if fork changes) ──────────────────────────
REPO_URL = "https://github.com/Sagelord00000001/Blackforge.git"
REPO_DIR = Path("/content/blackforge")
# ───────────────────────────────────────────────────────────────────────

if REPO_DIR.exists() and (REPO_DIR / "blackforge" / "__init__.py").exists():
    print(f"Repository already exists at {REPO_DIR}, updating...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")


---


In [ ]:
import subprocess

try:
    commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("Commit:", commit)
except Exception as e:
    print("Commit unavailable (expected in scratch checkouts):", e)


---


In [ ]:
!pip install hatchling --quiet
!pip install -e ".[dev]" --quiet


---


In [ ]:
import importlib

modules = [
    "blackforge",
    "blackforge.core.config",
    "blackforge.core.errors",
    "blackforge.core.types",
    "blackforge.runtime.bootstrap",
    "blackforge.memory",
    "blackforge.evidence",
    "blackforge.world_model",
    "blackforge.world_model.models",
    "blackforge.world_model.canonical",
    "blackforge.world_model.rules",
    "blackforge.world_model.query",
    "blackforge.world_model.repository",
    "blackforge.world_model.store",
    "blackforge.world_model.materializer",
    "blackforge.mission.manager",
    "blackforge.capabilities.registry",
    "blackforge.authorization",
    "blackforge.scope.models",
    "blackforge.scope.validator",
    "blackforge.recon",
    "blackforge.recon.models",
    "blackforge.recon.mock",
    "blackforge.recon.normalization",
    "blackforge.recon.evidence",
    "blackforge.recon.materializer",
    "blackforge.recon.capabilities",
    "blackforge.recon.engine",
    "blackforge.webapi",
    "blackforge.webapi.models",
    "blackforge.webapi.mock",
    "blackforge.webapi.normalization",
    "blackforge.webapi.evidence",
    "blackforge.webapi.capabilities",
    "blackforge.webapi.engine",
    "blackforge.webapi.materializer",
    "blackforge.webapi.redaction",
]

_import_failures = []
for module in modules:
    try:
        importlib.import_module(module)
    except Exception as e:
        _import_failures.append((module, str(e)))

if _import_failures:
    for mod, err in _import_failures:
        print(f"  FAIL: {mod} — {err}")
    raise RuntimeError(f"Import health check failed: {len(_import_failures)} module(s)")

print(f"Blackforge imports OK ({len(modules)} modules verified).")
print("Web API module imports: PASS")


---


In [ ]:
import subprocess
import sys

print("Running automated test suite...")
# The LLM/torch-heavy files are excluded: importing the HF provider pulls
# ~2GB of torch memory and can SIGKILL the kernel on CPU runtimes. Those
# tests are validated locally and in the Phase 1 notebook.
result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "--tb=short",
        "--ignore=tests/test_huggingface_provider.py",
        "--ignore=tests/test_loader.py",
        "--ignore=tests/test_smoke_real_model.py",
    ],
    capture_output=True, text=True, cwd=str(REPO_DIR),
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "")
    raise RuntimeError(f"pytest failed with exit code {result.returncode}")

print("Automated test suite: PASS")


---


In [ ]:
import os
from pathlib import Path

DBROOT = Path("data/phase6_colab").resolve()
DBROOT.mkdir(parents=True, exist_ok=True)
os.environ["BLACKFORGE_DB_PATH"] = str(DBROOT / "blackforge.db")
os.environ["BLACKFORGE_MEMORY_DB_PATH"] = str(DBROOT / "memory.db")
os.environ["BLACKFORGE_EVIDENCE_DB_PATH"] = str(DBROOT / "evidence.db")
os.environ["BLACKFORGE_WORLD_MODEL_DB_PATH"] = str(DBROOT / "world_model.db")
for _p in (DBROOT / "evidence.db", DBROOT / "world_model.db"):
    _p.unlink(missing_ok=True)

from blackforge.runtime.bootstrap import bootstrap

app = bootstrap()
assert app.healthy(), "Blackforge health check failed"
verification = app.verify()
for key in ("evidence_store_ready", "evidence_memory_link_ready", "memory_ready",
            "world_model_ready", "recon_ready", "webapi_ready"):
    assert verification[key], f"{key} must be True"

BOOTSTRAP_OK = app.healthy() and verification["webapi_ready"]

for k, v in verification.items():
    symbol = "PASS" if v else "FAIL"
    print(f"  [{symbol}] {k}")

print("\nBlackforge bootstrap (webapi ready): PASS")


---


In [ ]:
from blackforge.webapi.models import WebApiRequest
from blackforge.scope.models import TargetScope, Target, detect_target_type
from blackforge.core.types import RiskLevel

def _target(value: str) -> Target:
    return Target(value=value, target_type=detect_target_type(value))

MID = "mission_phase6_webapi"
DEMO_HOSTS = [
    "web.example.com", "www.example.com", "api.example.com",
    "mail.example.com", "throttled.example.com", "unreachable.example.com",
]
scope = TargetScope(
    mission_id=MID,
    allowed_targets=[_target(t) for t in DEMO_HOSTS],
    max_risk_level=RiskLevel.HIGH,
)
req = WebApiRequest(mission_id=MID, scope=scope, max_observations=2000)

engine = app.webapi_engine
assert engine is not None and len(engine.capabilities) == 10
expected = sorted([
    "webapi.application_discovery",
    "webapi.endpoint_enumeration",
    "webapi.api_surface_discovery",
    "webapi.security_header_analysis",
    "webapi.cookie_analysis",
    "webapi.cors_analysis",
    "webapi.auth_surface_observation",
    "webapi.openapi_review",
    "webapi.graphql_discovery",
    "webapi.request_response_observation",
])
ids_seen = sorted(c.capability_id for c in engine.capabilities)
assert ids_seen == expected, (ids_seen, expected)
print("Registered webapi capabilities:", ", ".join(c.capability_id for c in engine.capabilities))

_meta_by_id = {c.capability_id: c.meta() for c in engine.capabilities}
for capability_id in expected:
    meta = _meta_by_id[capability_id]
    print(
        f"  {meta.id:<34} risk={meta.risk_level.value:<6} mode={meta.mode.value:<7} "
        f"targets={[t.value for t in meta.supported_target_types]}"
    )
CAPS_OK = True

res = engine.discover_web_applications(req, "web.example.com")
assert res.status.value == "success"
assert res.capability_id == "webapi.application_discovery"
print(
    f"\napplication_discovery -> {res.observations[0].host} : "
    f"{res.observations[0].title} [{res.observations[0].technologies}]"
)


---


In [ ]:
from blackforge.evidence.models import EvidenceRelation
from blackforge.world_model.query import RelationshipQuery
from blackforge.world_model.models import EntityType

r_app = engine.discover_web_applications(req, "api.example.com")
r_ep = engine.enumerate_endpoints(req, "web.example.com")
r_api = engine.identify_api_surfaces(req, "api.example.com")
r_hdr = engine.inspect_security_headers(req, "api.example.com")
r_ck = engine.inspect_cookies(req, "web.example.com")
r_cors = engine.analyze_cors(req, "www.example.com")
r_auth = engine.inspect_authentication(req, "api.example.com")
r_oa = engine.parse_openapi(req, "api.example.com")
r_gql = engine.discover_graphql(req, "api.example.com")
r_rr = engine.observe_request_response(req, "api.example.com")

kinds = sorted({o.kind for r in (r_app, r_ep, r_api, r_hdr, r_ck, r_cors, r_auth, r_oa, r_gql, r_rr) for o in r.observations})
print("Pipeline observation kinds observed:", ", ".join(kinds))
for expected_kind in ("application", "endpoint", "api", "security_header", "cookie",
                      "cors", "auth_surface", "openapi", "graphql", "request_response"):
    assert expected_kind in kinds, expected_kind

print(
    "Request/response observation:",
    r_rr.observations[0].method, r_rr.observations[0].url,
    "->", r_rr.observations[0].status_code,
)

# Every observation evidence row is DERIVED_FROM its run's artifact row.
rel_ok = True
count_obs = 0
count_artifacts = 0
runs = (r_app, r_ep, r_api, r_hdr, r_ck, r_cors, r_auth, r_oa, r_gql, r_rr)
for r in runs:
    artifact = r.evidence_ids[0]
    count_artifacts += 1
    for ev_id in r.evidence_ids[1:]:
        rels = app.evidence_store.get_relationships(ev_id)
        ok = any(
            x.relation_type == EvidenceRelation.DERIVED_FROM
            and str(x.target_id) == str(artifact)
            for x in rels
        )
        rel_ok = rel_ok and ok
        count_obs += 1
assert rel_ok and count_obs >= 10
print(f"DERIVED_FROM links: {count_obs} observations across {count_artifacts} artifacts")

# World model: APPLICATION named by hostname; ENDPOINT named by URL.
app_entity = engine.world_model.find_entity(MID, EntityType.APPLICATION, "api.example.com")
assert app_entity is not None and app_entity.properties.get("url") == "https://api.example.com/"
ep_entity = engine.world_model.find_entity(MID, EntityType.ENDPOINT, "https://api.example.com/v1/health")
assert ep_entity is not None, "ENDPOINT entity must exist"
print("APPLICATION:", app_entity.name, "| ENDPOINT:", ep_entity.name)

# Analysis assertions bound to the host APPLICATION entity; request/response to ENDPOINT.
app_assertions = engine.world_model.list_assertions(str(app_entity.id))
ep_assertions = engine.world_model.list_assertions(str(ep_entity.id))
assert len(app_assertions) > 0 and len(ep_assertions) > 0
print(f"APPLICATION-bound assertions: {len(app_assertions)} | ENDPOINT-bound assertions: {len(ep_assertions)}")

# CONTAINS relationships exist; no attack-graph relationship types.
rels = engine.world_model.list_relationships(RelationshipQuery(mission_id=MID, limit=1000))
contains = [r for r in rels if getattr(r.relationship_type, "value", r.relationship_type) == "contains"]
assert len(contains) > 0, "CONTAINS relationships must exist"
attack_types = {"EXPLOITS", "CAN_COMPROMISE", "LEADS_TO", "ENABLES"}
rel_types = {getattr(r.relationship_type, "value", r.relationship_type) for r in rels}
assert not (rel_types & attack_types), f"Attack-graph types found: {rel_types & attack_types}"
print(f"CONTAINS relationships: {len(contains)} | No attack-graph relationship types: PASS")


---


In [ ]:
# Mission isolation: work under a second mission is fully disjoint.
MID2 = "mission_phase6_webapi_other"
scope2 = TargetScope(
    mission_id=MID2,
    allowed_targets=[_target("web.example.com")],
    max_risk_level=RiskLevel.HIGH,
)
req2 = WebApiRequest(mission_id=MID2, scope=scope2)
res2 = engine.enumerate_endpoints(req2, "web.example.com")
other_ids = {str(x) for x in res2.evidence_ids}
assert other_ids.isdisjoint({str(x) for x in r_ep.evidence_ids})
assert app.evidence_store.count(MID2) == len(res2.evidence_ids)
assert engine.world_model.count_entities(MID2) >= 1
print("Mission isolation: second mission produced its own evidence/world rows: PASS")

# Redaction: secret-like values are never persisted in plaintext.
from blackforge.webapi.redaction import redact_secret, redact_headers, redact_document
for secret in ("mock-bearer", "mock-session-web", "mock-password"):
    assert secret not in r_rr.raw_output, f"Leaked {secret} in raw output"
h = redact_headers({"Authorization": "Bearer mock-bearer", "Server": "nginx"})
assert h["Authorization"].startswith("REDACTED:") and h["Server"] == "nginx"
assert redact_secret("secret") == __import__("hashlib").sha256(b"secret").hexdigest()
print("Redaction at the boundary: PASS")


---


In [ ]:
from blackforge.core.errors import AuthorizationError, WebApiExecutionError

# 1) Target outside the scope is denied BEFORE any transport runs.
denied_out = False
try:
    engine.discover_web_applications(req, "scanme.example.org")
except AuthorizationError:
    denied_out = True
assert denied_out
print("Out-of-scope target denied before transport execution: PASS")

# 2) Unknown / non-typed capability is rejected.
unknown_rejected = False
try:
    engine.run(req, "webapi.not_real", "web.example.com")
except WebApiExecutionError:
    unknown_rejected = True
assert unknown_rejected
print("Unknown capability rejected (no generic execution surface): PASS")

# 3) Failure states: NO_EVIDENCE / RATE_LIMITED / REQUEST_FAILED / LIMITED.
res = engine.discover_web_applications(req, "mail.example.com")
assert res.status.value == "no_evidence" and not res.observations and res.warnings
res = engine.discover_web_applications(req, "throttled.example.com")
assert res.status.value == "rate_limited" and "rate limited" in (res.error or "")
res = engine.enumerate_endpoints(req, "unreachable.example.com")
assert res.status.value == "request_failed" and "connection refused" in (res.error or "")
limited_req = WebApiRequest(mission_id=MID, scope=scope, max_observations=2)
res = engine.inspect_security_headers(limited_req, "api.example.com")
assert res.status.value == "limited" and len(res.observations) == 2
assert any("limit" in w for w in res.warnings)
print("Failure states (NO_EVIDENCE, RATE_LIMITED, REQUEST_FAILED, LIMITED): PASS")


---


In [ ]:
from blackforge.evidence.repository import SQLiteEvidenceRepository
from blackforge.evidence.store import EvidenceStore
from blackforge.world_model.repository import SQLiteWorldRepository
from blackforge.world_model.store import WorldModelStore

# Fresh connections over the same SQLite files prove restart persistence.
fresh_ev = EvidenceStore(SQLiteEvidenceRepository(str(DBROOT / "evidence.db")))
fresh_wm = WorldModelStore(SQLiteWorldRepository(str(DBROOT / "world_model.db")))

persisted_ev = fresh_ev.count(MID) == app.evidence_store.count(MID)
persisted_wm = fresh_wm.count_entities(MID) == engine.world_model.count_entities(MID)
api_entity = fresh_wm.find_entity(MID, EntityType.APPLICATION, "api.example.com")
persisted_api = (
    api_entity is not None
    and api_entity.properties.get("url") == "https://api.example.com/"
)
assert persisted_ev and persisted_wm and persisted_api
rel_count = len(fresh_wm.list_relationships(RelationshipQuery(mission_id=MID, limit=1000)))
assert rel_count > 0
assert fresh_ev.count(MID) > 0
PERSIST_OK = persisted_ev and persisted_wm and persisted_api

for store in (fresh_ev, fresh_wm):
    store.close()
try:
    app.evidence_store.close()
except Exception:
    pass
try:
    app.world_model.close()
except Exception:
    pass
print("Restart persistence (fresh connections on same DB files): PASS")
print("Backends closed. Validation summary below.")


---


In [ ]:
results = {}
phase_checks = {
    "repository_integrity": (REPO_DIR / "blackforge" / "webapi" / "engine.py").exists(),
    "phase6_modules": bool(
        (REPO_DIR / "blackforge" / "webapi" / "capabilities.py").exists()
        and (REPO_DIR / "blackforge" / "webapi" / "evidence.py").exists()
        and (REPO_DIR / "blackforge" / "webapi" / "materializer.py").exists()
    ),
    "imports": len(_import_failures) == 0,
    "bootstrap_webapi_ready": BOOTSTRAP_OK,
    "no_generic_executor": CAPS_OK,
    "capability_surface": CAPS_OK,
    "pipeline_evidence": rel_ok,
    "world_materialized": app_entity is not None and ep_entity is not None,
    "assertions_bound": len(app_assertions) > 0 and len(ep_assertions) > 0,
    "no_attack_graph": not (rel_types & attack_types),
    "scope_authorization": denied_out,
    "unknown_capability_rejected": unknown_rejected,
    "mission_isolation": bool(other_ids.isdisjoint({str(x) for x in r_ep.evidence_ids})),
    "restart_persistence": PERSIST_OK,
}

# The pytest cell aborts the run on failure, so reaching this cell proves it passed.
pytest_passed = True
install_ok = len(_import_failures) == 0

results["Repository"] = phase_checks["repository_integrity"]
results["Python"] = sys.version_info >= (3, 10)
results["Hardware"] = True  # CPU fallback always works; this notebook needs no GPU
results["Installation"] = install_ok
results["Imports"] = install_ok
results["Automated tests"] = pytest_passed
results["Bootstrap"] = phase_checks["bootstrap_webapi_ready"]
results["Phase-specific tests"] = all(phase_checks.values())
results["Security checks"] = phase_checks["scope_authorization"] and phase_checks["unknown_capability_rejected"]

print()
print("=" * 60)
print("PHASE 6 COLAB VALIDATION SUMMARY")
print("=" * 60)
for name, ok in results.items():
    symbol = "PASS" if ok else "FAIL"
    print(f"  [{symbol}] {name}")

_all_ok = all(results.values()) and all(phase_checks.values())
assert _all_ok, "One or more validation checks failed"

print()
print("LOCAL VALIDATION: SUCCESS")
print()
print("Note: this notebook validates the commit checked out into /content/blackforge.")
